# dLEM quick-start: H1-hESC chr10

This notebook fits a **dLEM** (Differentiable Loop Extrusion Model) to a small
H1-hESC Hi-C dataset and reproduces the reference locus contact map from the paper
at **chr10:20.5–22.5 Mb** (10 kb resolution).

**Data**: 700 bins (7 Mb) extracted from `4DNFI9GMP2J8` (H1-hESC, 4D Nucleome),
stored as `data/example_chr10.cool`. The chromosome is named `ref_region` with
coordinates starting at 0; bin 0 corresponds to chr10:19,000,000.

**To run without a Jupyter server** (headless, from the terminal):
```bash
# option A — run the companion Python script directly
python docs/quick_start.py

# option B — execute this notebook headlessly
jupyter execute docs/quick_start.ipynb
```
Both produce `docs/quick_start_patch.png`.

In [ ]:
import os
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import jax.numpy as jnp

from dlem.api import fetch_band, train_dlem, flip_diag_row
from dlem.core import normalize_expected_observed, jax_forward_generate

## 1. Load data

`example_chr10.cool` contains the raw contact counts and ICE balancing weights
for chr10:19–26 Mb. `fetch_band` applies adaptive coarse-graining and NaN filling,
returning a (700 diagonals × 700 bins) band matrix.

In [ ]:
COOL       = 'data/example_chr10.cool'
REGION     = 'ref_region:0-7000000'
RESOLUTION = 10_000

# Offset for converting local bin positions to true chr10 coordinates (Mb)
GENOMIC_OFFSET_MB = 19.0

band = fetch_band(COOL, RESOLUTION, REGION, width=700)  # (700, 700)
band_train = band[1:170, :]                              # rows 1–169 (exclude row 0)

print(f'band shape: {band.shape},  dtype: {band.dtype}')

## 2. Train dLEM

## 3. Generate prediction for the reference locus

The example region is **chr10:20.5–22.5 Mb**, which falls at local bins 150–350
within the 7 Mb window.

## 3. Generate prediction for the reference locus

The reference locus shown in the paper is **chr10:20.5–22.5 Mb**, which falls at
local bins 150–350 within the 7 Mb window.

In [ ]:
PATCH_START = 150   # local bin → chr10:20.5 Mb
PATCH_SPAN  = 200   # 200 bins × 10 kb = 2 Mb → chr10:20.5–22.5 Mb

pred_band = np.array(jax_forward_generate(
    jnp.array(p_left [PATCH_START:PATCH_START + PATCH_SPAN], jnp.float32),
    jnp.array(p_right[PATCH_START:PATCH_START + PATCH_SPAN], jnp.float32),
    0.025,
    PATCH_SPAN,
))
obs_band = band[:PATCH_SPAN, PATCH_START:PATCH_START + PATCH_SPAN]

def _log_eo(b):
    return np.array(normalize_expected_observed(jnp.asarray(b, jnp.float32)))

pred_sq  = flip_diag_row(_log_eo(pred_band))
obs_sq   = flip_diag_row(_log_eo(obs_band))
# Upper triangle = observation, lower triangle = prediction
combined = np.triu(obs_sq) + np.tril(pred_sq.T, k=-1)

## 5. Plot

## 5. Plot (paper style)

In [ ]:
cmap = sns.color_palette('vlag', as_cmap=True)

patch_start_mb = GENOMIC_OFFSET_MB + PATCH_START * RESOLUTION / 1e6   # 20.5
patch_end_mb   = patch_start_mb + PATCH_SPAN * RESOLUTION / 1e6       # 22.5

# Bin centres in Mb — aligned with matshow extent
coords  = np.linspace(patch_start_mb, patch_end_mb, PATCH_SPAN, endpoint=False)
L_patch = p_left [PATCH_START:PATCH_START + PATCH_SPAN]
R_patch = p_right[PATCH_START:PATCH_START + PATCH_SPAN]

fig = plt.figure(figsize=(6, 6))
gs = fig.add_gridspec(
    2, 2,
    hspace=0, wspace=0,
    height_ratios=[1, 5], width_ratios=[5, 1],
)
ax_main  = fig.add_subplot(gs[1, 0])
ax_top   = fig.add_subplot(gs[0, 0], sharex=ax_main)
ax_right = fig.add_subplot(gs[1, 1], sharey=ax_main)

# Contact map — extent sets real Mb coordinates on both axes
im = ax_main.matshow(
    combined, cmap=cmap, vmin=-2, vmax=2,
    extent=[patch_start_mb, patch_end_mb, patch_end_mb, patch_start_mb],
)
ax_main.xaxis.set_ticks_position('bottom')
ax_main.tick_params(labelsize=8)
ax_main.set_xlabel('chr10 (Mb)', fontsize=9)
ax_main.set_ylabel('chr10 (Mb)', fontsize=9)
ax_main.text(0.97, 0.97, 'obs',  fontsize=8, ha='right', va='top',
             transform=ax_main.transAxes, color='white')
ax_main.text(0.03, 0.03, 'pred', fontsize=8, ha='left',  va='bottom',
             transform=ax_main.transAxes, color='white')

# L track (top) — shares x-axis with contact map
ax_top.plot(coords, L_patch, lw=1.5, c='tab:blue')
ax_top.set_ylim(0, 1.05)
ax_top.set_yticks([0, 0.5, 1])
ax_top.set_yticklabels(['', '', '1'], fontsize=7)
ax_top.set_ylabel('L', fontsize=9)
ax_top.xaxis.set_visible(False)
ax_top.set_title('chr10:20.5–22.5 Mb  |  H1-hESC 10 kb', fontsize=9)

# R track (right) — shares y-axis with contact map
ax_right.plot(R_patch, coords, lw=1.5, c='tab:orange')
ax_right.set_xlim(0, 1.05)
ax_right.set_xticks([0, 0.5, 1])
ax_right.set_xticklabels(['', '', '1'], fontsize=7)
ax_right.set_xlabel('R', fontsize=9)
ax_right.xaxis.set_ticks_position('top')
ax_right.xaxis.set_label_position('top')
ax_right.yaxis.set_visible(False)

# Force exact alignment — gridspec panels can drift slightly due to tick-label rendering
fig.canvas.draw()
main_bbox  = ax_main.get_position()
top_bbox   = ax_top.get_position()
right_bbox = ax_right.get_position()
total_w    = right_bbox.x0 + right_bbox.width - main_bbox.x0
right_w    = total_w / 6   # width_ratios [5, 1] → R panel is 1/6 of total
ax_top.set_position([main_bbox.x0, top_bbox.y0, main_bbox.width, top_bbox.height])
ax_right.set_position([main_bbox.x0 + main_bbox.width, main_bbox.y0,
                        right_w, main_bbox.height])

# Colorbar in its own axes to the right of the R panel — does not steal from it
cbar_ax = fig.add_axes([
    main_bbox.x0 + main_bbox.width + right_w + 0.01,
    main_bbox.y0,
    0.025,
    main_bbox.height,
])
fig.colorbar(im, cax=cbar_ax, label='log(obs/exp)')

plt.savefig('quick_start_patch.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: quick_start_patch.png')